In [1]:
import numpy as np
import pandas as pd
from scipy import stats
import plotly.express as px
from datetime import datetime, timedelta

In [2]:
# Read the data into a data frame from Parquet
df = pd.read_parquet("extract-3-very-clean.parquet")

In [4]:
# See how many records are included (rows)
len(df.index)

7177044

In [5]:
# Date fields are loaded from Parquet; verify types
df.dtypes

Property ID                              str
Sale counter                             str
Download date / time                     str
Property name                            str
Property unit number                     str
Property house number                    str
Property street name                     str
Property locality                        str
Property post code                   float64
Area                                 float64
Area type                                str
Contract date                 datetime64[us]
Settlement date               datetime64[us]
Purchase price                       float64
Zoning                                   str
Nature of property                       str
Primary purpose                          str
Strata lot number                        str
Dealing number                           str
Property legal description               str
dtype: object

In [1]:
# Filter the dataset to your own search area
# (could filter by whatever, but this is my search area)

property_locations = ['Raglan']
property_streetname = None  # e.g.: ['Railway Ave']
exclude_zoning = ['IN1', 'IN2', 'I', 'B', 'B1', 'B2', 'B7']
exclude_primary_purpose = ['Service stations', 'Service stati', 'Service statio', 'Shop', 'Hall', 'Commercial']
include_only_primary_purpose = None  # e.g.: ['Vacant land']
postcode_min = 2750
postcode_max = 2800
price_min = None
price_max = 5000000
area_min = 100
area_max = None
start_date = '1990-01-01'
end_date = '2100-01-01'

# Implement all filters
df_myarea = df.copy()
if property_locations is not None:
    df_myarea = df_myarea[df_myarea['Property locality'].isin(property_locations)]
if property_streetname is not None:
    df_myarea = df_myarea[df_myarea['Property street name'].isin(property_streetname)]
if area_min is not None:
    df_myarea = df_myarea[df_myarea['Area'] >= area_min]
if area_max is not None:
    df_myarea = df_myarea[df_myarea['Area'] <= area_max]
if postcode_min is not None:
    df_myarea = df_myarea[df_myarea['Property post code'] >= postcode_min]
if postcode_max is not None:
    df_myarea = df_myarea[df_myarea['Property post code'] <= postcode_max]
if price_min is not None:
    df_myarea = df_myarea[df_myarea['Purchase price'] >= price_min]
if price_max is not None:
    df_myarea = df_myarea[df_myarea['Purchase price'] <= price_max]
if exclude_zoning is not None:
    df_myarea = df_myarea[~df_myarea['Zoning'].isin(exclude_zoning)]
if start_date is not None:
    df_myarea = df_myarea[df_myarea['Contract date'] >= pd.to_datetime(start_date)]
if end_date is not None:
    df_myarea = df_myarea[df_myarea['Contract date'] <= pd.to_datetime(end_date)]
if exclude_primary_purpose is not None:
    df_myarea = df_myarea[~df_myarea['Primary purpose'].isin(exclude_primary_purpose)]
if include_only_primary_purpose is not None:
    df_myarea = df_myarea[df_myarea['Primary purpose'].isin(include_only_primary_purpose)]

print(str(len(df_myarea.index)) + ' records kept')

NameError: name 'df' is not defined

In [7]:
#Show zoning and purpose types in the dataset
#Types: https://www.valuergeneral.nsw.gov.au/__data/assets/pdf_file/0019/216406/Property_Sales_Data_File_Zone_Codes_and_Descriptions_V2.pdf

display(df_myarea['Primary purpose'].unique())
display(df_myarea['Zoning'].unique())

<ArrowStringArray>
[             nan,           'Null',      'Residence',    'Vacant Land',
  'House/Townhou',     'Town House',   'Fire Station',      'Townhouse',
          'Flats',     'Farm Rural',          'Motel',           'Road',
  'Commercial Pr',    'Residential',         'Garage',  'Service Stati',
  'Retirement Vi',         'School',         'Church',         'Vacant',
          'Units',          'House',        'Factory',         'Retail',
      'Structure', 'Shop And Resid', 'Retail/Commerc', 'Vac Land & Res',
   'Masonic Hall',  'Factory Units', 'Service Statio',         'Office']
Length: 32, dtype: str

<ArrowStringArray>
[ 'A',  'R',  nan,  'P',  'S',  'O',  'X', 'R2', 'E4', 'E3', 'R3', 'E2', 'C4',
 'C3', 'C2', 'E1']
Length: 16, dtype: str

In [8]:
# Fix NaNs without chained-assignment side effects
df_myarea = df_myarea.assign(
    Zoning=df_myarea['Zoning'].fillna('None'),
    Area=df_myarea['Area'].fillna(0),
    **{'Purchase price': df_myarea['Purchase price'].fillna(0)}
)

In [9]:
#Remove purchase price outliers

before=len(df_myarea.index)

#Display the outliers
display(df_myarea[(np.abs(stats.zscore(df_myarea['Purchase price'])) >= 3)])

#Remove them from the data
#df_myarea = df_myarea[(np.abs(stats.zscore(df_myarea['Purchase price'])) < 3)]

after=len(df_myarea.index)
print('Removed ' + str(before-after) + ' outliers (more than 3 standard deviations from the mean).')

,Property ID,Sale counter,Download date / time,Property name,Property unit number,Property house number,Property street name,Property locality,Property post code,Area,Area type,Contract date,Settlement date,Purchase price,Zoning,Nature of property,Primary purpose,Strata lot number,Dealing number,Property legal description
178326,2269361.0,NaN,NaN,NaN,NaN,26,Perry Av,Springwood,2777.0,1296.0,M,1991-12-20,NaT,1730000.0,A,NaN,NaN,NaN,NaN,LOT 45 DP 27823.
863536,2268288.0,NaN,NaN,NaN,NaN,79,Hawkesbury Rd,Springwood,2777.0,18270.0,H,1995-08-15,NaT,3581920.0,A,NaN,NaN,NaN,NaN,LOT 2 DP 532226.
1443535,2271259.0,NaN,NaN,NaN,NaN,24,Spurwood Rd,Warrimoo,2774.0,8037.0,M,1998-03-27,NaT,1641000.0,A,NaN,NaN,NaN,NaN,LOT 63 DP 13669.
1823115,2268650.0,NaN,NaN,NaN,NaN,17,Linksview Rd,Springwood,2777.0,1214.0,M,2000-12-06,NaT,2400000.0,A,NaN,NaN,NaN,NaN,LOT 106 DP 31559.
1823757,2269915.0,NaN,NaN,NaN,NaN,8,Thomson Av,Springwood,2777.0,834.7,M,2000-09-14,NaT,2050000.0,A,NaN,NaN,NaN,NaN,LOT 17 DP 26810.
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7166560,2250286,36,20260126 01:08,NaN,NaN,108,Grose Rd,Faulconbridge,2776.0,1462.0,M,2025-11-28,2026-01-19,1400000.0,C4,R,Residence,NaN,AV802870,2/260542
7166569,2266809,45,20260126 01:08,NaN,NaN,18,Banjo Pl,Springwood,2777.0,1524.0,M,2025-10-24,2026-01-20,1610000.0,C4,R,Residence,NaN,AV805909,233/803316
7171317,2271457,19,20260202 01:08,NaN,NaN,12,The Terrace,Warrimoo,2774.0,1897.0,M,2025-12-10,2026-01-28,1400000.0,C4,R,Residence,NaN,AV828405,7/717880
7171327,2269632,29,20260202 01:08,NaN,NaN,50,Raymond Rd,Springwood,2777.0,2757.0,M,2025-12-01,2026-01-23,1610000.0,C4,R,Residence,NaN,AV817213,1/215933


Removed 0 outliers (more than 3 standard deviations from the mean).


In [11]:
#Price histogram in ~$50K bins

if not df_myarea.empty:
    fig = px.histogram(df_myarea, x="Purchase price", nbins=int(df_myarea['Purchase price'].max()/50000),
        title='Price histogram', width=1000, height=400,
    )
    fig.show()


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [12]:
#Show the top 5 most expensive properties in the area

df_myarea.sort_values('Purchase price', ascending=False).head(5)

,Property ID,Sale counter,Download date / time,Property name,Property unit number,Property house number,Property street name,Property locality,Property post code,Area,Area type,Contract date,Settlement date,Purchase price,Zoning,Nature of property,Primary purpose,Strata lot number,Dealing number,Property legal description
3045058,2826401,78,20061216 01:17,NaN,1,575,Great Western Hwy,Faulconbridge,2776.0,1180.0,M,2006-11-21,2006-11-29,4730000.0,None,3,Service Stati,1.0,AC779924,1/SP68928
3565629,2261043,35,20081124 01:16,NaN,NaN,3,Sulman Rd,Lawson,2783.0,33020.0,H,2008-08-29,2008-08-29,4200000.0,A,3,School,NaN,AE289012,"B/30715 3, 107/32238"
3565612,2254160,18,20081124 01:16,NaN,NaN,62,Hall Pde,Hazelbrook,2779.0,38010.0,H,2008-08-29,2008-08-29,4200000.0,A,3,School,NaN,AE289012,150/712935
6814041,2861300,26,20240722 01:06,NaN,NaN,10,Railway Pde,Springwood,2777.0,11070.0,H,2023-11-13,2024-07-15,3900000.0,A,R,Residence,NaN,AU246591,2/739780 11/1033926
863536,2268288.0,NaN,NaN,NaN,NaN,79,Hawkesbury Rd,Springwood,2777.0,18270.0,H,1995-08-15,NaT,3581920.0,A,NaN,NaN,NaN,NaN,LOT 2 DP 532226.


In [13]:
#Show all properties with a purchase price of 0 (these are likely missing values)

df_myarea[df_myarea['Purchase price'] == 0]

,Property ID,Sale counter,Download date / time,Property name,Property unit number,Property house number,Property street name,Property locality,Property post code,Area,Area type,Contract date,Settlement date,Purchase price,Zoning,Nature of property,Primary purpose,Strata lot number,Dealing number,Property legal description
89677,2254117.0,NaN,NaN,NaN,NaN,37,Hall Pde,Hazelbrook,2779.0,1884.0,M,1991-08-09,NaT,0.0,A,NaN,NaN,NaN,NaN,LOT 119 DP 32238.
1875187,2250813.0,35.0,20010719 12:15,NaN,NaN,40,Russell Ave,Faulconbridge,2776.0,2201.0,M,2000-05-18,2000-05-18,0.0,A,R,NaN,NaN,7519028,58//8712
1875190,2250861.0,38.0,20010719 12:15,NaN,NaN,61,St Georges Cres,Faulconbridge,2776.0,929.0,M,2000-10-23,2000-10-23,0.0,A,R,NaN,NaN,7545222,3//16532
1875193,2251164.0,41.0,20010719 12:15,NaN,NaN,23,Taronga Way,Faulconbridge,2776.0,1011.0,M,2001-01-08,2001-01-08,0.0,A,R,NaN,NaN,7552967,13//836233
1875194,2251167.0,42.0,20010719 12:15,NaN,NaN,17,Taronga Way,Faulconbridge,2776.0,1074.0,M,2001-03-12,2001-03-12,0.0,A,3,Null,NaN,7538107,10//836233
1875313,2267681.0,161.0,20010719 12:15,NaN,NaN,7,De Chair Ave,Springwood,2777.0,765.1,M,2000-10-12,2000-10-12,0.0,A,R,NaN,NaN,7510644,1//27705
1875314,2267792.0,162.0,20010719 12:15,NaN,NaN,15,Ellison Rd,Springwood,2777.0,657.6,M,2000-06-21,2000-06-21,0.0,A,R,NaN,NaN,7559502,1//390915
1875316,2268019.0,164.0,20010719 12:15,NaN,NaN,44,Farm Rd,Springwood,2777.0,1132.0,M,2001-02-20,2001-02-20,0.0,A,R,NaN,NaN,7558339,3//521727
1875318,2268433.0,166.0,20010719 12:15,NaN,NaN,58,Huntley Grange Rd,Springwood,2777.0,1024.0,M,2001-03-06,2001-03-06,0.0,A,R,NaN,NaN,7558667,20//27035
1875323,2269326.0,171.0,20010719 12:15,NaN,NaN,32,Paterson Rd,Springwood,2777.0,834.7,M,1992-04-03,1992-04-03,0.0,A,R,NaN,NaN,7549688,3//223990


In [ ]:
#Display all the records.
display(df_myarea)

In [ ]:
# Price by size and contract date

# Scale property size so dots don't get too small
median = df_myarea['Area'].median()
df_myarea = df_myarea.copy()
df_myarea['Area - scaled'] = ((df_myarea['Area'] - median) / 1000) + median

fig = px.scatter(
    df_myarea,
    x='Contract date',
    y='Purchase price',
    size='Area - scaled',
    color='Zoning',
    title='Price and size of property by contract date',
    width=1000,
    height=500,
    labels={'x': 'Contract date'},
    hover_name=df_myarea['Property house number'] + ' ' + df_myarea['Property street name'] + ', ' + df_myarea['Property locality'],
    hover_data={
        'Area - scaled': False,
        'Zoning': True,
        'Area': True
    }
)

fig.show()

In [ ]:
#Price by contract date

fig = px.scatter(
    df_myarea,
    x='Contract date',
    y='Purchase price',    
    title='Price over time',
    trendline='rolling',
    trendline_options=dict(window=45),    
    trendline_color_override="red",
    width=1000,
    height=500,
    labels={'x':'Contract date'},
    hover_name=df_myarea['Property house number'] + ' ' + df_myarea['Property street name'] + ', ' + df_myarea['Property locality'],
    hover_data={
        'Area - scaled':False,
        'Zoning':True,
        'Area':True
    }
)

fig.show()

In [ ]:
#Median price by contract date

df_myarea_agg=df_myarea[['Contract date','Purchase price']]
df_myarea_agg=df_myarea_agg.groupby(['Contract date']).median()
#This is the same as above:
##df_myarea_agg=df_myarea_agg.groupby(([pd.Grouper(key='Contract date', freq='D')])).median()

fig = px.scatter(
    df_myarea_agg,
    x=df_myarea_agg.index.values,
    y='Purchase price',    
    title='Median price by date',
    width=1000,
    height=500,
    labels={'x':'Contract date'},
)

fig.show()

In [ ]:
# Get the marker date (~90 days before last completed week)
offset = -datetime.now().weekday() - 7 - 90
this_date = datetime.now() + timedelta(offset)
print(this_date)

In [ ]:
#Monthly median price

df_myarea_aggM = df_myarea[['Contract date', 'Purchase price']]
df_myarea_aggM = df_myarea_aggM.groupby([pd.Grouper(key='Contract date', freq='ME')]).agg('median')

df_myarea_aggM['Rolling 6-month median'] = df_myarea_aggM.rolling(6).median()

#Could also do this if we wanted to show multiple types - e.g. mean, sum, etc
#g1 = df_myarea_m.groupby(pd.Grouper(key='Contract date', freq="M")).median()
#g2 = df_myarea_m.groupby(pd.Grouper(key='Contract date', freq="M")).mean()
#g = g1.merge(g2, left_on='Contract date', right_on='Contract date', suffixes=(' median', ' mean'))

fig = px.line(
    df_myarea_aggM,
    title='Monthly median purchase price',
    width=1000,
    height=500
)

fig.add_vline(x=this_date, line_width=2, line_dash="dot", line_color="green")
fig.show()

In [ ]:
# Sales volume by month

latest_date = df_myarea['Contract date'].max() - timedelta(days=90)

df_myarea_aggMc = df_myarea[['Contract date', 'Purchase price']]
df_myarea_aggMc = df_myarea_aggMc.groupby([pd.Grouper(key='Contract date', freq='ME')]).agg('count')
df_myarea_aggMc = df_myarea_aggMc.rename(columns={'Purchase price': 'Number of sales'})
df_myarea_aggMc['Rolling 6-month median'] = df_myarea_aggMc.rolling(6).median()

fig = px.line(
    df_myarea_aggMc,
    title='Sales volume by month',
    width=1000,
    height=500
)

fig.add_vline(x=this_date, line_width=2, line_dash='dot', line_color='green')
fig.show()